In [3]:
#%pip install xgboost

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(DATA_URL)
df = df.fillna(0)
print('Dataset shape:', df.shape)
print(df.head(3).to_string(index=False))

Dataset shape: (10000, 11)
 model_year origin fuel_type        drivetrain  num_doors  engine_displacement  num_cylinders  horsepower  vehicle_weight  acceleration  fuel_efficiency_mpg
       2006 Europe  Gasoline Front-wheel drive          4                 2180              6       243.0            3870           0.0                 31.9
       2008 Europe    Diesel Front-wheel drive          4                 2390              6       272.0            4210           0.0                 31.3
       1996   Asia  Gasoline Front-wheel drive          5                 2320              6       267.0            4240          17.3                 27.5


In [2]:
target = 'fuel_efficiency_mpg'

# Exact random splits specified in the 2026 homework.
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

features = [c for c in df.columns if c != target]

def to_dicts(frame):
    return frame[features].to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(to_dicts(df_train))
X_val = dv.transform(to_dicts(df_val))
y_train = df_train[target].to_numpy()
y_val = df_val[target].to_numpy()
feature_names = dv.get_feature_names_out()

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print('Train/val/test rows:', len(df_train), len(df_val), len(df_test))
print('Encoded feature count:', len(feature_names))


Train/val/test rows: 6000 2000 2000
Encoded feature count: 16


# Q1

In [5]:
tree = DecisionTreeRegressor(max_depth=1, random_state=1)
tree.fit(X_train, y_train)
first_feature = feature_names[tree.tree_.feature[0]]
original_feature = first_feature.split('=')[0]
print(original_feature)

model_year


# Q2

In [8]:
rf10 = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf10.fit(X_train, y_train)
q2 = rmse(y_val, rf10.predict(X_val))
print(q2)

1.843032473940706


# Q3

In [12]:
forest_sizes = [10, 50, 100, 150]
q3_results = []
for n_trees in forest_sizes:
    model = RandomForestRegressor(n_estimators=n_trees, random_state=1, n_jobs=-1)
    model.fit(X_train, y_train)
    score = rmse(y_val, model.predict(X_val))
    q3_results.append({'n_estimators': n_trees, 'RMSE': score, 'RMSE_3dp': round(score, 3)})
q3_table = pd.DataFrame(q3_results)
display(q3_table)
best_q3 = q3_table.loc[q3_table['RMSE_3dp'].idxmin(), 'n_estimators']
print(int(best_q3))

,n_estimators,RMSE,RMSE_3dp
0,10,1.843032,1.843
1,50,1.773650,1.774
2,100,1.768922,1.769
3,150,1.768196,1.768


150


# Q4

In [15]:
depths = [10, 15, 20, 25]
q4_results = []
for depth in depths:
    for n_trees in forest_sizes:
        model = RandomForestRegressor(max_depth=depth, n_estimators=n_trees,
                                      random_state=1, n_jobs=-1)
        model.fit(X_train, y_train)
        score = rmse(y_val, model.predict(X_val))
        q4_results.append({'max_depth': depth, 'n_estimators': n_trees, 'RMSE': score})
q4_table = pd.DataFrame(q4_results)
mean_by_depth = q4_table.groupby('max_depth', as_index=True)['RMSE'].mean()
print('Mean RMSE by max_depth:')
print(mean_by_depth.to_string())
print(int(mean_by_depth.idxmin()))


Mean RMSE by max_depth:
max_depth
10    1.755597
15    1.783092
20    1.786768
25    1.787995
10


# Q5

In [19]:
rf_imp = RandomForestRegressor(n_estimators=10, max_depth=20,
                               random_state=1, n_jobs=-1)
rf_imp.fit(X_train, y_train)
importance = pd.Series(rf_imp.feature_importances_, index=feature_names)
choices = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
print(importance.reindex(choices).idxmax())

vehicle_weight


# Q6

In [21]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(feature_names))
dval = xgb.DMatrix(X_val, label=y_val, feature_names=list(feature_names))
watchlist = [(dtrain, 'train'), (dval, 'val')]

base_params = {
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'nthread': 8,
    'seed': 1,
    'verbosity': 0,
}
q6_scores = {}
for eta in [0.3, 0.1]:
    params = {**base_params, 'eta': eta}
    history = {}
    booster = xgb.train(params, dtrain, num_boost_round=100,
                        evals=watchlist, evals_result=history, verbose_eval=False)
    score = rmse(y_val, booster.predict(dval))
    q6_scores[eta] = score
    print(f'eta = {eta}: final validation RMSE = {score:.6f}; '
          f'best during training = {min(history["val"]["rmse"]):.6f}')

    q6_answer = min(q6_scores, key=q6_scores.get)
print(q6_answer)

eta = 0.3: final validation RMSE = 1.826458; best during training = 1.732048
eta = 0.1: final validation RMSE = 1.724810; best during training = 1.712171
0.1
